# `apptbl-vaccines-slopes`: Vaccines OLS treatment effects by attribute

In [1]:
from pathlib import Path
import sys

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ASSET_NAME = "apptbl-vaccines-slopes"
ASSET_DIR = Path.cwd()
PAPER_DIR = ASSET_DIR.parent
if not (PAPER_DIR / "microdata_analysis.py").is_file():
    raise FileNotFoundError(f"Open this notebook from manuscript-training-assets/; {PAPER_DIR} has no microdata_analysis.py")
if str(PAPER_DIR) not in sys.path:
    sys.path.insert(0, str(PAPER_DIR))

import microdata_analysis as ma

print(f"Outputs go to {ASSET_DIR / ASSET_NAME}.*")

Outputs go to /home/dw/github/decision-models-finetuning/code/train-notebooks/beta-training/ms_paper_beta_loss/manuscript-training-assets/apptbl-vaccines-slopes.*


## Run design

In [2]:
RUN_ROOTS = (PAPER_DIR / "runs-rorqual", PAPER_DIR / "runs-nibi")
MODELS = ("kev-0.8b", "kev-4b", "kev-9b")
BASE_SLUG = "row1_base_untrained"
TRAINED_SLUGS = ("row2b_wu_ce_oracle", "row3_berke_ce_only", "row5_full_berke_wu_ce_oracle")
VALIDATORS = ("kriner2026", "grummon2025")

ctx = ma.load_microdata_context(
    run_roots=RUN_ROOTS,
    models=MODELS,
    base_slug=BASE_SLUG,
    trained_slugs=TRAINED_SLUGS,
    validators=VALIDATORS,
)
display(ctx.status_grid)

,kev-0.8b,kev-4b,kev-9b
regime,,,
row1_base_untrained,step 0 [test_best ✓],step 0 [test_best ✓],step 0 [test_best ✓]
row2b_wu_ce_oracle,finished (@640/1395) [test_best ✓],finished (@416/1395) [test_best ✓],finished (@928/1395) [test_best ✓]
row3_berke_ce_only,finished (@1152/2730) [test_best ✓],finished (@544/2730) [test_best ✓],finished (@256/2730) [test_best ✓]
row5_full_berke_wu_ce_oracle,finished (@1600/4125) [test_best ✓],finished (@1472/4125) [test_best ✓],finished (@1600/4125) [test_best ✓]


## Labels and formatting (edit wording here)

In [3]:
BETAS_FIELD = "covid_betas_by_model"
FIRST_COLUMN_HEADER = "Vaccine attribute level"
HUMAN_HEADER = "Human"
MODEL_SHORT_LABELS = {"kev-0.8b": "0.8B", "kev-4b": "4B", "kev-9b": "9B"}
COMPARED_REGIMES = {BASE_SLUG: "Base", "row5_full_berke_wu_ce_oracle": "Pooled"}

# Optional renames, raw term -> text shown in the table. Terms not listed are shown as they are.
TERM_LABELS = {}

# Summary rows under the slopes: label -> which terms ("all", "price", "non-price"). {n} becomes the number of terms.
SUMMARY_ROWS = {r"$\mathrm{MSE}_\beta \times 10^3$": "all"}
PRICE_MARKER = "| Price:"

SLOPE_DECIMALS = 3
MSE_DECIMALS = 2
MSE_SCALE = 1000.0

## Rows (terms) in table order

In [4]:
terms = list(getattr(ctx, BETAS_FIELD)[MODELS[0]][BASE_SLUG].index)
print(f"{len(terms)} terms")

11 terms


## Build the table

In [5]:
def escape_markdown(text):
    """Escape $ and | so Pandoc reads them as text inside a pipe table."""
    escaped = text.replace("$", r"\$")
    escaped = escaped.replace("|", r"\|")
    return escaped


def pipe_table(headers, rows, column_rules):
    """Pandoc pipe table: header row, rule row (e.g. ':---' left, '---:' right), then one line per row."""
    if len(column_rules) != len(headers):
        raise ValueError(f"{len(column_rules)} column rules for {len(headers)} columns")
    lines = ["| " + " | ".join(headers) + " |"]
    lines.append("|" + "|".join(column_rules) + "|")
    for row in rows:
        if len(row) != len(headers):
            raise ValueError(f"Row has {len(row)} cells, expected {len(headers)}: {row}")
        lines.append("| " + " | ".join(row) + " |")
    return "\n".join(lines) + "\n"

In [6]:
human_slopes = getattr(ctx, BETAS_FIELD)[MODELS[0]][BASE_SLUG].loc[terms, "human_pop_beta"].astype(float)

compared_columns = []
compared_slopes = []
for model_slug in MODELS:
    for regime_slug, regime_label in COMPARED_REGIMES.items():
        betas = getattr(ctx, BETAS_FIELD)[model_slug][regime_slug]
        compared_columns.append(f"{MODEL_SHORT_LABELS[model_slug]} {regime_label}")
        compared_slopes.append(betas.loc[terms, "model_pop_beta"].astype(float))

headers = [FIRST_COLUMN_HEADER, HUMAN_HEADER, *compared_columns]
rows = []
for term in terms:
    shown_term = TERM_LABELS[term] if term in TERM_LABELS else term
    row = [escape_markdown(shown_term), f"{human_slopes.loc[term]:.{SLOPE_DECIMALS}f}"]
    for slopes in compared_slopes:
        row.append(f"{slopes.loc[term]:.{SLOPE_DECIMALS}f}")
    rows.append(row)

for row_label_template, term_group in SUMMARY_ROWS.items():
    if term_group not in ("all", "price", "non-price"):
        raise ValueError(f"Unknown term group {term_group!r}")
    selected_terms = []
    for term in terms:
        is_price = PRICE_MARKER in term
        if term_group == "all":
            selected_terms.append(term)
        elif term_group == "price" and is_price:
            selected_terms.append(term)
        elif term_group == "non-price" and not is_price:
            selected_terms.append(term)
    if not selected_terms:
        raise ValueError(f"No terms selected for summary row {row_label_template!r}")
    row_label = row_label_template.replace("{n}", str(len(selected_terms)))
    row = [f"**{row_label}**", f"{0.0:.{MSE_DECIMALS}f}"]
    for slopes in compared_slopes:
        errors = slopes.loc[selected_terms] - human_slopes.loc[selected_terms]
        mse = float((errors ** 2).mean()) * MSE_SCALE
        row.append(f"**{mse:.{MSE_DECIMALS}f}**")
    rows.append(row)

column_rules = [":---"] + ["---:"] * (len(headers) - 1)
table_md = pipe_table(headers, rows, column_rules)
display(pd.DataFrame(rows, columns=headers))

,Vaccine attribute level,Human,0.8B Base,0.8B Pooled,4B Base,4B Pooled,9B Base,9B Pooled
0,Efficacy: 70% vs 50%,0.074,0.042,0.056,0.125,0.109,0.211,0.154
1,Efficacy: 90% vs 50%,0.164,0.065,0.104,0.238,0.211,0.369,0.289
2,Duration: 5y vs 1y,0.054,0.017,0.023,0.005,0.038,0.025,0.046
3,Severe side effects: 1/1M vs 1/10k,0.065,0.009,0.024,0.024,0.045,0.050,0.054
4,Mild side effects: 1/30 vs 1/10,0.012,-0.000,-0.009,0.008,0.018,0.016,0.020
5,FDA: EUA vs Full approval,-0.029,0.034,0.017,-0.068,-0.039,-0.008,-0.013
6,Origin: UK vs US,-0.038,-0.025,-0.030,-0.081,-0.053,-0.072,-0.051
7,Origin: China vs US,-0.131,0.028,-0.004,-0.197,-0.114,-0.205,-0.143
8,Endorsed: Biden vs Trump,0.018,0.109,0.063,0.050,0.046,0.118,0.040
9,Endorsed: CDC vs Trump,0.089,0.273,0.081,0.284,0.138,0.306,0.123


## Save `apptbl-vaccines-slopes.md` (included by the manuscript)

In [7]:
table_path = ASSET_DIR / f"{ASSET_NAME}.md"
table_path.write_text(table_md, encoding="utf-8")
print(table_md)
print(f"Saved {table_path}")

| Vaccine attribute level | Human | 0.8B Base | 0.8B Pooled | 4B Base | 4B Pooled | 9B Base | 9B Pooled |
|:---|---:|---:|---:|---:|---:|---:|---:|
| Efficacy: 70% vs 50% | 0.074 | 0.042 | 0.056 | 0.125 | 0.109 | 0.211 | 0.154 |
| Efficacy: 90% vs 50% | 0.164 | 0.065 | 0.104 | 0.238 | 0.211 | 0.369 | 0.289 |
| Duration: 5y vs 1y | 0.054 | 0.017 | 0.023 | 0.005 | 0.038 | 0.025 | 0.046 |
| Severe side effects: 1/1M vs 1/10k | 0.065 | 0.009 | 0.024 | 0.024 | 0.045 | 0.050 | 0.054 |
| Mild side effects: 1/30 vs 1/10 | 0.012 | -0.000 | -0.009 | 0.008 | 0.018 | 0.016 | 0.020 |
| FDA: EUA vs Full approval | -0.029 | 0.034 | 0.017 | -0.068 | -0.039 | -0.008 | -0.013 |
| Origin: UK vs US | -0.038 | -0.025 | -0.030 | -0.081 | -0.053 | -0.072 | -0.051 |
| Origin: China vs US | -0.131 | 0.028 | -0.004 | -0.197 | -0.114 | -0.205 | -0.143 |
| Endorsed: Biden vs Trump | 0.018 | 0.109 | 0.063 | 0.050 | 0.046 | 0.118 | 0.040 |
| Endorsed: CDC vs Trump | 0.089 | 0.273 | 0.081 | 0.284 | 0.138 | 0.306 | 0